In [29]:
import pandas as pd

In [30]:
icu_labels = pd.read_csv('label.csv')
print("Number of labels:", len(icu_labels))
print("Number of unique stay_ids:", len(icu_labels['stay_id'].unique()))
print("Number of unique subject_ids:", len(icu_labels['subject_id'].unique()))

Number of labels: 202531
Number of unique stay_ids: 202531
Number of unique subject_ids: 202524


/tmp/ipykernel_40018/2727390277.py:1: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  icu_labels = pd.read_csv('label.csv')


In [31]:
icu_labels

,subject_id,hadm_id,stay_id,intime,outtime,outcome_icu_transfer_12h,intime_icu,Trigger_time,Sample_time,Time
0,11054342,28706109.0,30016443,2187-12-23 09:57:00,2187-12-24 00:14:00,True,2187-12-24 00:14:00,14.283333,NaN,14.283333
1,11858684,24200796.0,30022721,2122-10-11 16:34:00,2122-10-12 06:45:00,True,2122-10-12 06:45:00,14.183333,NaN,14.183333
2,17856950,23029492.0,30036859,2134-04-13 00:16:00,2134-04-13 17:57:00,True,2134-04-13 17:57:00,17.683333,NaN,17.683333
3,19770161,21099154.0,30044128,2184-03-06 12:31:00,2184-03-07 15:01:00,True,2184-03-07 15:01:00,26.500000,NaN,26.500000
4,12966433,24995627.0,30071358,2163-08-07 11:14:00,2163-08-07 22:54:00,True,2163-08-07 22:54:00,11.666667,NaN,11.666667
...,...,...,...,...,...,...,...,...,...,...
202526,10157595,23785188.0,39999863,2122-01-16 18:02:00,2122-01-16 23:01:00,False,NaN,NaN,3.938374,3.938374
202527,17060831,NaN,39999933,2128-07-20 13:42:00,2128-07-20 19:49:00,False,NaN,NaN,3.818404,3.818404
202528,17255106,23020029.0,39999939,2155-08-04 11:15:00,2155-08-05 12:48:00,False,NaN,NaN,8.220081,8.220081
202529,15768202,NaN,39999953,2152-06-22 14:08:00,2152-06-22 18:13:00,False,NaN,NaN,19.079299,19.079299


In [32]:
demo = pd.read_csv('demo.csv')
print("Number of unique subject_ids in demo:", len(demo['subject_id'].unique()))
print("Number of unique subject_ids in icu_labels:", len(icu_labels['subject_id'].unique()))
print("Number of unique subject_ids in demo and icu_labels:", len(set(demo['subject_id']).intersection(set(icu_labels['subject_id']))))

Number of unique subject_ids in demo: 205504
Number of unique subject_ids in icu_labels: 202524
Number of unique subject_ids in demo and icu_labels: 202524


In [33]:
meds = pd.read_csv('med.csv')

In [34]:
# keep only the subject_id, stay_id, charttime, and name columns
meds = meds[['subject_id', 'stay_id', 'charttime', 'name']]
# rename 'name' to Text
meds = meds.rename(columns={'name': 'Text'})

In [35]:
numerics = pd.read_csv('numerics.csv')
numerics_per_subject = numerics.groupby('stay_id').size()
print("Median number of meds per subject_id:", numerics_per_subject.median())
print("Mean number of meds per subject_id:", numerics_per_subject.mean())
print("max number of meds per subject_id:", numerics_per_subject.max())

Median number of meds per subject_id: 4.0
Mean number of meds per subject_id: 5.049712176566209
max number of meds per subject_id: 110


In [36]:
# in numerics, create a new column 'Text' from 'temperature', 'heartrate', 'resprate', 'o2sat', 'sbp', 'dbp', 'acuity'
# e.g. Temp: 36.5, HR: 80, RR: 20, O2Sat: 98, SBP: 120, DBP: 80, Acuity: 1
# but dont include the text for an item if not nan

numerics['temperature'] = numerics['temperature'].apply(lambda x: f'Temp: {x}, ' if pd.notna(x) else '')
numerics['heartrate'] = numerics['heartrate'].apply(lambda x: f'HR: {x}, ' if pd.notna(x) else '')
numerics['resprate'] = numerics['resprate'].apply(lambda x: f'RR: {x} ,' if pd.notna(x) else '')
numerics['o2sat'] = numerics['o2sat'].apply(lambda x: f'O2Sat: {x}, ' if pd.notna(x) else '')
numerics['sbp'] = numerics['sbp'].apply(lambda x: f'SBP: {x}, ' if pd.notna(x) else '')
numerics['dbp'] = numerics['dbp'].apply(lambda x: f'DBP: {x}, ' if pd.notna(x) else '')
numerics['acuity'] = numerics['acuity'].apply(lambda x: f'Acuity: {x}' if pd.notna(x) else '')
# now combine all these columns into a single column 'Text', separated by commas
numerics['Text'] = numerics['temperature'] + numerics['heartrate']  + numerics['resprate']  + numerics['o2sat'] + numerics['sbp'] + numerics['dbp'] + numerics['acuity']


In [37]:
# keep the stay_id, charttime, and Text columns
numerics = numerics[['stay_id', 'charttime', 'Text']]

In [38]:
vitals = pd.read_csv('vitals.csv')

In [39]:
# count median number of vitals per subject_id
vitals_per_subject = vitals.groupby('stay_id').size()
print("Median number of vitals per subject_id:", vitals_per_subject.median())
print("Mean number of vitals per subject_id:", vitals_per_subject.mean())
print("max number of vitals per subject_id:", vitals_per_subject.max())

Median number of vitals per subject_id: 4.0
Mean number of vitals per subject_id: 4.680681836894565
max number of vitals per subject_id: 110


In [40]:
# remove all rows in vitals where rhythm is nan
vitals = vitals[vitals['rhythm'].notna()]
# remove all rows where rhythm is 'Sinus Rhythm' or 'Normal Sinus Rhythm'
vitals = vitals[vitals['rhythm'] != 'Sinus Rhythm']
vitals = vitals[vitals['rhythm'] != 'Normal Sinus Rhythm']

In [41]:
# get the unique values and their counts in the rhythm column
rhythm_counts = vitals['rhythm'].value_counts()
print("Number of unique rhythms:", len(rhythm_counts))
rhythm_counts

Number of unique rhythms: 1167


rhythm
Atrial Fibrillation             5238
Sinus Tachycardia               5133
Sinus Bradycardia               3246
sr                              3020
Paced Rhythm                    1777
                                ... 
nsr with ? incomplete bbb          1
98/51                              1
second degree av block             1
130/61                             1
?afib vs nsr with freq pac's       1
Name: count, Length: 1167, dtype: int64

In [42]:
# rename the rhythm column to 'Text'
vitals = vitals.rename(columns={'rhythm': 'Text'})
# keep only the subject_id, stay_id, charttime, and Text columns
vitals = vitals[['stay_id', 'charttime', 'Text']]

In [43]:
# demo['age'] = 'Age: ' + demo['age'].astype(str)
# # rename 'age' to 'Text'
# demo = demo.rename(columns={'age': 'Text'})
# # keep only the subject_id, stay_id, and Text columns
# demo = demo[['subject_id', 'stay_id', 'Text']]

In [44]:
print("Number of rows in numerics:", len(numerics))
# drop duplicates in numerics where stay_id and charttime are the same
numerics = numerics.drop_duplicates(subset=['stay_id', 'charttime'])
print("Number of rows in numerics after dropping duplicates:", len(numerics))

Number of rows in numerics: 2146567
Number of rows in numerics after dropping duplicates: 1911262


In [45]:
all_data = pd.concat([meds, numerics, vitals], ignore_index=True)

In [46]:
# sort all_data by stay_id and charttime
all_data = all_data.sort_values(by=['stay_id', 'charttime'])

In [47]:
# merge all_data with labels on stay_id
all_data = all_data.merge(icu_labels, on='stay_id', how='inner')

In [48]:
# relative time in hours = charttime - intime
all_data['charttime'] = pd.to_datetime(all_data['charttime'])
all_data['intime'] = pd.to_datetime(all_data['intime'])
all_data['relative_time_hrs'] = (all_data['charttime'] - all_data['intime']).dt.total_seconds() / 3600

all_data

,subject_id_x,stay_id,charttime,Text,subject_id_y,hadm_id,intime,outtime,outcome_icu_transfer_12h,intime_icu,Trigger_time,Sample_time,Time,relative_time_hrs
0,NaN,30000039,2165-10-06 11:47:00,"Temp: 98.6, HR: 85.0, RR: 16.0 ,O2Sat: 98.0, S...",13340997,23100190.0,2165-10-06 11:47:00,2165-10-06 20:18:00,False,NaN,NaN,6.007971,6.007971,0.000000
1,NaN,30000039,2165-10-06 13:10:00,"Temp: 98.1, HR: 80.0, RR: 18.0 ,O2Sat: 95.0, S...",13340997,23100190.0,2165-10-06 11:47:00,2165-10-06 20:18:00,False,NaN,NaN,6.007971,6.007971,1.383333
2,NaN,30000039,2165-10-06 14:46:00,"Temp: 97.4, HR: 86.0, RR: 18.0 ,O2Sat: 95.0, S...",13340997,23100190.0,2165-10-06 11:47:00,2165-10-06 20:18:00,False,NaN,NaN,6.007971,6.007971,2.983333
3,NaN,30000039,2165-10-06 15:52:00,"SBP: 176.0, DBP: 112.0,",13340997,23100190.0,2165-10-06 11:47:00,2165-10-06 20:18:00,False,NaN,NaN,6.007971,6.007971,4.083333
4,NaN,30000039,2165-10-06 16:54:00,"HR: 96.0, RR: 22.0 ,O2Sat: 98.0, SBP: 165.0, D...",13340997,23100190.0,2165-10-06 11:47:00,2165-10-06 20:18:00,False,NaN,NaN,6.007971,6.007971,5.116667
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1546812,NaN,39999939,2155-08-05 06:28:00,SR,17255106,23020029.0,2155-08-04 11:15:00,2155-08-05 12:48:00,False,NaN,NaN,8.220081,8.220081,19.216667
1546813,NaN,39999939,2155-08-05 12:47:00,"Temp: 98.1, HR: 78.0, RR: 15.0 ,O2Sat: 99.0, S...",17255106,23020029.0,2155-08-04 11:15:00,2155-08-05 12:48:00,False,NaN,NaN,8.220081,8.220081,25.533333
1546814,NaN,39999953,2152-06-22 14:08:00,"Temp: 98.2, HR: 108.0, RR: 18.0 ,O2Sat: 100.0,...",15768202,NaN,2152-06-22 14:08:00,2152-06-22 18:13:00,False,NaN,NaN,19.079299,19.079299,0.000000
1546815,NaN,39999953,2152-06-22 17:30:00,"Temp: 98.3, HR: 81.0, RR: 16.0 ,O2Sat: 100.0, ...",15768202,NaN,2152-06-22 14:08:00,2152-06-22 18:13:00,False,NaN,NaN,19.079299,19.079299,3.366667


In [49]:
# keep only the stay_id, subject_id_y, relative_time_hrs, Time and Text columns
all_data = all_data[['stay_id', 'subject_id_y', 'relative_time_hrs', 'Text', 'Time']]
all_data

,stay_id,subject_id_y,relative_time_hrs,Text,Time
0,30000039,13340997,0.000000,"Temp: 98.6, HR: 85.0, RR: 16.0 ,O2Sat: 98.0, S...",6.007971
1,30000039,13340997,1.383333,"Temp: 98.1, HR: 80.0, RR: 18.0 ,O2Sat: 95.0, S...",6.007971
2,30000039,13340997,2.983333,"Temp: 97.4, HR: 86.0, RR: 18.0 ,O2Sat: 95.0, S...",6.007971
3,30000039,13340997,4.083333,"SBP: 176.0, DBP: 112.0,",6.007971
4,30000039,13340997,5.116667,"HR: 96.0, RR: 22.0 ,O2Sat: 98.0, SBP: 165.0, D...",6.007971
...,...,...,...,...,...
1546812,39999939,17255106,19.216667,SR,8.220081
1546813,39999939,17255106,25.533333,"Temp: 98.1, HR: 78.0, RR: 15.0 ,O2Sat: 99.0, S...",8.220081
1546814,39999953,15768202,0.000000,"Temp: 98.2, HR: 108.0, RR: 18.0 ,O2Sat: 100.0,...",19.079299
1546815,39999953,15768202,3.366667,"Temp: 98.3, HR: 81.0, RR: 16.0 ,O2Sat: 100.0, ...",19.079299


In [50]:
# get the first row of every stay_id
icu_labels['Time'].describe()

count    202531.000000
mean          5.246042
std           5.158365
min           0.500618
25%           3.041048
50%           4.455763
75%           6.466667
max         493.896513
Name: Time, dtype: float64

In [51]:
all_data['EndTime'] = all_data['Time'] - 6

/tmp/ipykernel_40018/3866449284.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  all_data['EndTime'] = all_data['Time'] - 6


In [52]:
# Keep only the rows where EndTime is positive
all_data = all_data[all_data['EndTime'] > 0]
all_data['EndTime'].describe()

count    477399.000000
mean          3.326501
std           7.887047
min           0.000077
25%           0.845125
50%           2.054872
75%           4.208855
max         487.896513
Name: EndTime, dtype: float64

In [53]:
# count number of unique stay_id in all_data
print("Number of unique stay_id in all_data:", len(all_data['stay_id'].unique()))

Number of unique stay_id in all_data: 59541


In [54]:
# keep the rows where relative_time_hrs <= EndTime
print("Number of rows before filtering:", len(all_data))
print("Number of unique stay_ids before filtering:", len(all_data['stay_id'].unique()))
all_data = all_data[all_data['relative_time_hrs'] <= all_data['EndTime']]
print("Number of rows after filtering:", len(all_data))
print("Number of unique stay_ids after filtering:", len(all_data['stay_id'].unique()))


Number of rows before filtering: 477399
Number of unique stay_ids before filtering: 59541
Number of rows after filtering: 207990
Number of unique stay_ids after filtering: 59541


In [97]:
# group by stay_id and get sizes
all_data_per_stay = all_data.groupby('stay_id').size()
print("Median number of rows per stay_id:", all_data_per_stay.median())
print("Mean number of rows per stay_id:", all_data_per_stay.mean())
print("Max number of rows per stay_id:", all_data_per_stay.max())

Median number of rows per stay_id: 2.0
Mean number of rows per stay_id: 3.4932231571522143
Max number of rows per stay_id: 58


In [98]:
# group by stay_id and get unique relative_time_hrs
all_data_per_stay = all_data.groupby('stay_id')['relative_time_hrs'].nunique()
print("Median number of unique relative_time_hrs per stay_id:", all_data_per_stay.median())
print("Mean number of unique relative_time_hrs per stay_id:", all_data_per_stay.mean())
print("Max number of unique relative_time_hrs per stay_id:", all_data_per_stay.max())

Median number of unique relative_time_hrs per stay_id: 2.0
Mean number of unique relative_time_hrs per stay_id: 2.9080633513041434
Max number of unique relative_time_hrs per stay_id: 52


In [ ]:
# how many stay_id have > 1 unique relative_time_hrs
all_data_per_stay = all_data.groupby('stay_id')['relative_time_hrs'].nunique()
print("Number of stay_id with > 1 unique relative_time_hrs:", len(all_data_per_stay[all_data_per_stay > 1]))

Number of stay_id with > 10 unique relative_time_hrs: 6464


In [100]:
# Keep only the stay_id with atleast 5 unique relative_time_hrs
all_data_per_stay = all_data.groupby('stay_id')['relative_time_hrs'].nunique()
all_data = all_data[all_data['stay_id'].isin(all_data_per_stay[all_data_per_stay > 5].index)]

In [101]:
# set precision of relative_time_hrs to 2 decimal places
all_data['relative_time_hrs'] = all_data['relative_time_hrs'].round(2)

In [135]:
def _condense_notes_per_csn(group: pd.DataFrame) -> str:
    """Helper to condense text notes for a patient_id group."""
    # For rows with the same time_difference, we want to combine them into a single entry
    group = group.groupby('relative_time_hrs').agg({
        'Text': lambda x: ";".join(x.astype(str))
    }).reset_index()

    return "\n".join(
        f"{row['relative_time_hrs']} - {row['Text']}"
        for _, row in group.astype({'Text': str, 'relative_time_hrs': str}).iterrows()
    )
# group by patient_id and relative_time_mins, and then condense the text notes
condensed_df = all_data.groupby('stay_id').apply(_condense_notes_per_csn).reset_index()
condensed_df.rename(columns={0: 'Text'}, inplace=True)

/tmp/ipykernel_3121969/255818858.py:13: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  condensed_df = all_data.groupby('stay_id').apply(_condense_notes_per_csn).reset_index()


In [136]:
print("Number of rows in condensed_df:", len(condensed_df))
print("number of unique stay_ids in condensed_df:", len(condensed_df['stay_id'].unique()))

Number of rows in condensed_df: 6464
number of unique stay_ids in condensed_df: 6464


In [137]:
# merge with icu_labels on stay_id
condensed_df = condensed_df.merge(icu_labels, on='stay_id', how='inner')

In [138]:
print("Number of rows in condensed_df:", len(condensed_df))
print("number of unique stay_ids in condensed_df:", len(condensed_df['stay_id'].unique()))

Number of rows in condensed_df: 6464
number of unique stay_ids in condensed_df: 6464


In [139]:
# remove duplicate stay_id in demo (keep the last one)
demo = demo.drop_duplicates(subset=['stay_id'], keep='last')
len(demo['stay_id'].unique())

425087

In [140]:
demo

,subject_id,stay_id,Text
0,11714491,30000012,Age: 66.0
1,14230614,30000017,Age: 42.0
2,13821532,30000038,Age: 82.0
3,13340997,30000039,Age: 80.0
4,19848164,30000055,Age: 64.0
...,...,...,...
425082,17255106,39999939,Age: 57.0
425083,15768202,39999953,Age: 37.0
425084,11302781,39999961,Age: 63.0
425085,16479007,39999964,Age: 58.0


In [141]:
# merge with demo on subject_id
condensed_df = condensed_df.merge(demo, on='stay_id', how='inner')   

In [142]:
print("Number of rows in condensed_df:", len(condensed_df))
print("number of unique stay_ids in condensed_df:", len(condensed_df['stay_id'].unique()))

Number of rows in condensed_df: 6464
number of unique stay_ids in condensed_df: 6464


In [143]:
# rename stay_id_x to stay_id, Text_x to Text, Text_y to Age
condensed_df = condensed_df.rename(columns={
    'stay_id_x': 'stay_id',
    'Text_x': 'Text',
    'Text_y': 'Age'
})
# combine the Text and Age columns
condensed_df['Text'] = condensed_df['Age'] + "\n" + condensed_df['Text']


In [144]:
# create a new column 'Label' that is true if Trigger_time is not nan
condensed_df['Label'] = condensed_df['Trigger_time'].notna()

In [146]:
# keep the stay_id, Text, and Label columns
condensed_df = condensed_df[['stay_id', 'Text', 'Label']]

In [147]:
# save to full.parquet
condensed_df.to_parquet('full.parquet', index=False)

In [148]:
pos_labels = condensed_df[condensed_df['Label'] == True]
print("Fraction of positive labels:", len(pos_labels) / len(condensed_df))

Fraction of positive labels: 0.2906868811881188
